# Chapter 3 Practical 02: Cosine, Pearson, and Jaccard Similarity

Learning objectives:
- Compute user-user similarity on co-rated items.
- Compare cosine and Pearson similarity for explicit ratings.
- Use Jaccard similarity for binary interactions.
- Use overlap counts to judge whether a similarity value is trustworthy.

Slide connection: similarity measures, cosine equation, Pearson equation, Jaccard similarity, and sparse overlap.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


The helper functions below compare users only on items that both users rated. Missing values stay missing; they are not treated as zero ratings.


In [ ]:
def common_ratings(matrix, user_a, user_b):
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    return pair.loc[user_a], pair.loc[user_b]

def cosine_on_overlap(matrix, user_a, user_b):
    a, b = common_ratings(matrix, user_a, user_b)
    if len(a) == 0:
        return np.nan
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return np.nan if denom == 0 else float(np.dot(a, b) / denom)

def pearson_on_overlap(matrix, user_a, user_b):
    a, b = common_ratings(matrix, user_a, user_b)
    if len(a) < 2:
        return np.nan
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def jaccard_liked(matrix, user_a, user_b, threshold=5):
    liked_a = set(matrix.columns[matrix.loc[user_a] >= threshold])
    liked_b = set(matrix.columns[matrix.loc[user_b] >= threshold])
    if not liked_a and not liked_b:
        return np.nan
    return len(liked_a & liked_b) / len(liked_a | liked_b)


In [ ]:
rows = []
target = "Karen"
for other in rating_matrix.index.drop(target):
    overlap_count = rating_matrix.loc[[target, other]].notna().all(axis=0).sum()
    rows.append({
        "target_user": target,
        "other_user": other,
        "co_rated_items": int(overlap_count),
        "cosine": cosine_on_overlap(rating_matrix, target, other),
        "pearson": pearson_on_overlap(rating_matrix, target, other),
        "jaccard_liked": jaccard_liked(rating_matrix, target, other),
    })

similarities = pd.DataFrame(rows).sort_values("pearson", ascending=False)
similarities.round(3)


Pearson removes each user's average rating behavior. This matters when one user rates generously and another rates strictly.


In [ ]:
def similarity_matrix(metric):
    users = rating_matrix.index
    sim = pd.DataFrame(index=users, columns=users, dtype=float)
    for u in users:
        for v in users:
            sim.loc[u, v] = metric(rating_matrix, u, v) if u != v else 1.0
    return sim

pearson_sim = similarity_matrix(pearson_on_overlap)
pearson_sim.round(2)


Small overlap can make a similarity score unstable. Here we keep the raw similarity and show the overlap count next to it so students can judge the evidence.


In [ ]:
similarities[["other_user", "co_rated_items", "cosine", "pearson", "jaccard_liked"]].round(3)


# Challenges

### Challenge 1 — Change the Target User

**Goal:**
Investigate whether a different target user has different nearest neighbors.

**What to do:**

1. Change the existing `target` variable from `"Karen"` to another user, such as `"Alice"` or `"Sally"`.
2. Rerun the similarity table.
3. Compare cosine, Pearson, Jaccard, and `co_rated_items`.
4. Identify which neighbors changed.


In [ ]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Which neighbors changed when you changed the target user? Did all similarity measures rank users in the same order? Which measure looked most reasonable?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 — Change the Jaccard Like Threshold

**Goal:**
Investigate how the definition of "liked" changes binary Jaccard similarity.

**What to do:**

1. In `jaccard_liked`, change `threshold=5` to `threshold=6`.
2. Rerun the similarity table for the same target user.
3. Compare the Jaccard values before and after the change.
4. Explain why some values changed more than others.


In [ ]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Which users had different Jaccard similarity values? Did cosine or Pearson change when the Jaccard threshold changed? Why?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 — Concept Check: Small Overlap

This challenge requires **no programming**.

Two users have a very high similarity score, but they have only one or two co-rated movies.

Explain why this similarity value may be less reliable than a slightly lower score based on many co-rated movies.

### Your explanation

> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
